# Study 832 — High-Yield Credit Momentum 📉

**Does the *trend* of high-yield credit tell you when to be long stocks?**

High-yield credit is a risk asset; its recent total return is a popular risk-on/off gauge.
Measured *duration-hedged* — **HYG in excess of IEF** — a positive trailing trend is read
as risk appetite rising (be long SPY) and a negative trend as time to de-risk (hold IEF).
We take the self-contained daily version on the four ETFs (2007-05-01 → 2026-06-30,
4,822 rows) and ask the two honest questions: does the credit trend **predict**
the equity leg, and can a **costed** SPY↔IEF timer beat buy-and-hold?

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Distinct from study 115 (credit-spread **level** warning), 795
(cross-sectional bond momentum), 131 (utilities canary).*


## 1. The idea in one picture

When high-yield bonds out-earn duration-matched Treasuries, the market is paying *up* for credit risk — risk appetite is rising. The folk rule: ride that into equities, and step aside into Treasuries when credit rolls over. It sounds like it should lead the stock market. Does it?

In [1]:
R = dict(diff_bps=-1.62, t_nw=-0.37, on_bps=2.99, off_bps=4.6, on_frac=0.636)
print('credit-trend risk-on days: SPY-over-IEF excess %+.2f bps/day' % R['on_bps'])
print('        risk-off days     : SPY-over-IEF excess %+.2f bps/day' % R['off_bps'])
print('difference (on - off)     : %+.2f bps/day  (NW t = %+.2f)' % (R['diff_bps'], R['t_nw']))
print('risk-on share of days     : %.0f%%' % (R['on_frac']*100))

credit-trend risk-on days: SPY-over-IEF excess +2.99 bps/day
        risk-off days     : SPY-over-IEF excess +4.60 bps/day
difference (on - off)     : -1.62 bps/day  (NW t = -0.37)
risk-on share of days     : 64%


## 2. Is the machinery even able to see a signal? A live synthetic control

We plant a real credit-times-equity effect in a seeded toy world (`edge>0`) and check the detector recovers it — and stays *silent* on the null (`edge=0`, credit trend present but not predictive). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from hy_credit_momentum import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=832+s, n_days=2200))['t_nw'] for s in range(6)])
planted = st.synthetic_detect(data.synthetic_panel(edge=0.006, seed=832, n_days=2200))
print('null worlds  : discrimination NW t mean %+.2f over 6 seeds  (should be ~0)' % null_t.mean())
print('planted world: discrimination NW t = %+.2f  (should light up)' % planted['t_nw'])

null worlds  : discrimination NW t mean +0.07 over 6 seeds  (should be ~0)
planted world: discrimination NW t = +2.99  (should light up)


## 3. The honest verdict — the credit trend does *not* time equities

On the real tape the risk-on days (positive credit trend) earned a SPY-over-IEF excess of **+2.99 bps/day** versus **+4.60 bps/day** on risk-off days — the *wrong* way round (difference **-1.62 bps/day**, NW *t* = **-0.37**), and the 3-month trend agrees (-2.55 bps, *t* = -0.61). A label-shift placebo puts the observed value at just -0.4σ (p = 0.65), and the sign **flips across eras** (+4.8 bps early, -12.3 bps late). The costed SPY↔IEF timer never beats buy-and-hold (net Sharpe 0.627 vs 0.619 at 1 bp, giving up 3.5%/yr of return) — it only trims the drawdown (-41% vs -55%). **Signal: None. Tradability: Mirage.**